# Stream Macroinvertebrates Image Classification
## Using MobileNet, EfficientNet and Vision Transformer Models

**Student ID:** U3243935  
**Unit:** Software Technology 1 (4483/8995)  
**Week:** 8 Tutorial & Computer Lab  

==================================

This notebook demonstrates how to:
1. Load a dataset organised by class folders (without predefined splits)
2. Perform train/validation split in code
3. Augment data during training
4. Build transfer learning models with **MobileNet**, **EfficientNet** and **Vision Transformer** pretrained on ImageNet
5. Train, evaluate, and predict on new images

In [ ]:
%cd /content/drive/MyDrive/2026_Teaching/ST1/Week_7_Tut_Lab_Activities

## Step 0: Data Preparation

1. Set runtime to GPU via Runtime > Change Runtime type
2. Mount Google Drive and set working folder
3. Download stream invertebrates dataset from Kaggle: https://www.kaggle.com/datasets/kennethtm/stream-macroinvertebrates/data
4. Create `insects_dataset` folder inside `Assets/`
5. Pick **three insect species** different from the demo (Asellus sp, Baetidae sp, Elmis sp)
6. Create `train_data` and `test_data` subfolders
7. Move 5 images per species into `test_data` (15 total)
8. Move remaining species folders into `train_data`

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image

DATASET_DIR = '/content/drive/MyDrive/2026_Teaching/ST1/Week_7_Tut_Lab_Activities/Assets/insects_dataset/train_data'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

## Step 1: Load dataset with 80/20 train-validation split

In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f'Classes found: {class_names}')

## Step 2: Optimise dataset performance

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

## Step 3: Data augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    augmented_images = data_augmentation(images)
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.show()

---
# 1. MobileNetV2 Model

In [ ]:
base_model_mobilenet = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model_mobilenet.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model_mobilenet(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

mobilenet_model = models.Model(inputs, outputs)
mobilenet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
mobilenet_model.summary()

In [ ]:
EPOCHS = 10
mobilenet_history = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

In [ ]:
acc = mobilenet_history.history['accuracy']
val_acc = mobilenet_history.history['val_accuracy']
loss = mobilenet_history.history['loss']
val_loss = mobilenet_history.history['val_loss']

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.legend()
plt.title('MobileNetV2 Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.legend()
plt.title('MobileNetV2 Loss')
plt.show()

val_loss_score, val_acc_score = mobilenet_model.evaluate(val_ds)
print(f'MobileNetV2 Validation accuracy: {val_acc_score*100:.2f}%')

---
# 2. EfficientNetB0 Model

In [ ]:
base_model_efficient = tf.keras.applications.EfficientNetB0(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model_efficient.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model_efficient(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

efficientnet_model = models.Model(inputs, outputs)
efficientnet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
efficientnet_model.summary()

In [ ]:
efficient_history = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

In [ ]:
acc = efficient_history.history['accuracy']
val_acc = efficient_history.history['val_accuracy']
loss = efficient_history.history['loss']
val_loss = efficient_history.history['val_loss']

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.legend()
plt.title('EfficientNetB0 Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.legend()
plt.title('EfficientNetB0 Loss')
plt.show()

val_loss_score, val_acc_score = efficientnet_model.evaluate(val_ds)
print(f'EfficientNetB0 Validation accuracy: {val_acc_score*100:.2f}%')

### Step 7 (Optional): Fine-tune EfficientNetB0

In [ ]:
base_model_efficient.trainable = True

fine_tune_at = 100
for layer in base_model_efficient.layers[:fine_tune_at]:
    layer.trainable = False

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

fine_tune_epochs = 10
total_epochs = EPOCHS + fine_tune_epochs

fine_history = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=efficient_history.epoch[-1]
)

acc_ft = efficient_history.history['accuracy'] + fine_history.history['accuracy']
val_acc_ft = efficient_history.history['val_accuracy'] + fine_history.history['val_accuracy']
loss_ft = efficient_history.history['loss'] + fine_history.history['loss']
val_loss_ft = efficient_history.history['val_loss'] + fine_history.history['val_loss']

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(acc_ft, label='Train Accuracy')
plt.plot(val_acc_ft, label='Val Accuracy')
plt.legend()
plt.title('Accuracy (including fine tuning)')

plt.subplot(1, 2, 2)
plt.plot(loss_ft, label='Train Loss')
plt.plot(val_loss_ft, label='Val Loss')
plt.legend()
plt.title('Loss (including fine tuning)')
plt.show()

---
# 3. Vision Transformer (ViT) Model

Using `keras-cv` or a custom lightweight ViT implementation.

In [ ]:
!pip install keras-cv --quiet

In [ ]:
import keras_cv

# Build ViT using keras_cv backbone
vit_backbone = keras_cv.models.ViTTiny16(
    include_rescaling=True,
    include_top=False,
    input_shape=IMG_SIZE + (3,)
)
vit_backbone.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = vit_backbone(x, training=False)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

vit_model = models.Model(inputs, outputs)
vit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
vit_model.summary()

In [ ]:
vit_history = vit_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

In [ ]:
acc = vit_history.history['accuracy']
val_acc = vit_history.history['val_accuracy']
loss = vit_history.history['loss']
val_loss = vit_history.history['val_loss']

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.legend()
plt.title('ViT Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.legend()
plt.title('ViT Loss')
plt.show()

val_loss_score, val_acc_score = vit_model.evaluate(val_ds)
print(f'ViT Validation accuracy: {val_acc_score*100:.2f}%')

---
# Model Comparison

In [ ]:
mobilenet_val_loss, mobilenet_val_acc = mobilenet_model.evaluate(val_ds, verbose=0)
efficient_val_loss, efficient_val_acc = efficientnet_model.evaluate(val_ds, verbose=0)
vit_val_loss, vit_val_acc = vit_model.evaluate(val_ds, verbose=0)

print('=== Model Comparison ===')
print(f'MobileNetV2  - Val Accuracy: {mobilenet_val_acc*100:.2f}%  Val Loss: {mobilenet_val_loss:.4f}')
print(f'EfficientNetB0 - Val Accuracy: {efficient_val_acc*100:.2f}%  Val Loss: {efficient_val_loss:.4f}')
print(f'ViT           - Val Accuracy: {vit_val_acc*100:.2f}%  Val Loss: {vit_val_loss:.4f}')

---
# Save Best Model & Run Inference

In [ ]:
best_acc = max(mobilenet_val_acc, efficient_val_acc, vit_val_acc)
if best_acc == mobilenet_val_acc:
    best_model = mobilenet_model
    print('Best model: MobileNetV2')
elif best_acc == efficient_val_acc:
    best_model = efficientnet_model
    print('Best model: EfficientNetB0')
else:
    best_model = vit_model
    print('Best model: Vision Transformer (ViT)')

best_model.save('macroinvertebrates_best_classifier.h5')
print("Model saved as 'macroinvertebrates_best_classifier.h5'")

## Step 10: Confusion matrix, precision, recall, F1 for best model

In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

y_true, y_pred, y_score = [], [], []

for images, labels in val_ds:
    preds = best_model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_score.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_score = np.array(y_score)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, colorbar=True)
ax.set_title('Confusion Matrix')
plt.show()

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
print(f'Weighted Precision: {precision:.4f}')
print(f'Weighted Recall:    {recall:.4f}')
print(f'Weighted F1-score:  {f1:.4f}')

y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
plt.figure(figsize=(8, 6))
for i, cls in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Class {cls} (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right', fontsize='small')
plt.show()

## Step 12: Inference on new test images

In [ ]:
TEST_IMAGE_DIR = '/content/drive/MyDrive/2026_Teaching/ST1/Week_7_Tut_Lab_Activities/Assets/insects_dataset/test_data'
os.makedirs(TEST_IMAGE_DIR, exist_ok=True)

def load_and_preprocess_image(img_path, img_size=IMG_SIZE):
    img = image.load_img(img_path, target_size=img_size)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

def predict_new_images(test_dir):
    img_files = [f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not img_files:
        print(f"No images found in '{test_dir}'. Please add some images and rerun.")
        return

    for img_file in img_files:
        path = os.path.join(test_dir, img_file)
        img_arr = load_and_preprocess_image(path)
        preds = best_model.predict(img_arr)
        pred_idx = np.argmax(preds[0])
        confidence = preds[0][pred_idx]

        print(f"Original Image: {img_file} --> Predicted: {class_names[pred_idx]} ({confidence*100:.2f}%)")
        img = image.load_img(path)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Prediction: {class_names[pred_idx]} ({confidence*100:.2f}%)")
        plt.show()

print("Running inference on images from 'test_data' folder...")
predict_new_images(TEST_IMAGE_DIR)